<a href="https://colab.research.google.com/github/deb8001422413/MOVIE-AND-MUSIC-RECOMMENDATION-SYSTEMS/blob/main/MOVIE_AND_MUSIC_RECOMMENDER_PY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [57]:
# ==========================
# PART 1
# Import Libraries
# ==========================

import numpy as np
import pandas as pd
import ast
import pickle

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import save_npz

print("NumPy :", np.__version__)
print("Pandas:", pd.__version__)

NumPy : 2.0.2
Pandas: 2.2.2


In [60]:
movies = pd.read_csv('/content/tmdb_5000_movies.csv.zip')
credits = pd.read_csv('/content/tmdb_5000_credits.csv.zip')

movies = movies.merge(credits, on='title')

movies = movies[['movie_id',
                 'title',
                 'overview',
                 'genres',
                 'keywords',
                 'cast',
                 'crew']]

movies.dropna(inplace=True)
movies.reset_index(drop=True, inplace=True)

movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [61]:
def convert(obj):

    if isinstance(obj, str):
        obj = ast.literal_eval(obj)

    L = []

    for i in obj:
        L.append(i['name'])

    return L

In [62]:
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast'] = movies['cast'].apply(convert)
movies['crew'] = movies['crew'].apply(convert)

In [63]:
movies['genres'] = movies['genres'].apply(
    lambda x:[i.replace(" ","") for i in x]
)

movies['keywords'] = movies['keywords'].apply(
    lambda x:[i.replace(" ","") for i in x]
)

movies['cast'] = movies['cast'].apply(
    lambda x:[i.replace(" ","") for i in x]
)

movies['crew'] = movies['crew'].apply(
    lambda x:[i.replace(" ","") for i in x]
)

In [64]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())

movies['tags'] = (
      movies['overview']
    + movies['genres']
    + movies['keywords']
    + movies['cast']
    + movies['crew']
)

In [65]:
movie_df = movies[['title','tags']]

movie_df['tags'] = movie_df['tags'].apply(
    lambda x:" ".join(x)
)

movie_df['tags'] = movie_df['tags'].apply(
    lambda x:x.lower()
)

movie_df.head()

/tmp/ipykernel_869/2114315138.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movie_df['tags'] = movie_df['tags'].apply(
/tmp/ipykernel_869/2114315138.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movie_df['tags'] = movie_df['tags'].apply(


,title,tags
0,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,Spectre,a cryptic message from bond’s past sends him o...
3,The Dark Knight Rises,following the death of district attorney harve...
4,John Carter,"john carter is a war-weary, former military ca..."


In [66]:
movies_dict = {
    "title": movie_df["title"].tolist()
}

pickle.dump(
    movies_dict,
    open("movies_dict.pkl","wb")
)

print("movies_dict.pkl Saved")

movies_dict.pkl Saved


In [67]:
# ==========================
# PART 2
# Movie Vectorization
# ==========================

from sklearn.feature_extraction.text import CountVectorizer
from scipy.sparse import save_npz
import pickle

cv_movie = CountVectorizer(
    max_features=5000,
    stop_words="english"
)

movie_vectors = cv_movie.fit_transform(movie_df["tags"])

print(movie_vectors.shape)

# Save sparse matrix
save_npz("movie_vectors.npz", movie_vectors)

# Save vocabulary
pickle.dump(cv_movie.vocabulary_, open("movie_vocab.pkl", "wb"))

print("movie_vectors.npz Saved")

(4806, 5000)
movie_vectors.npz Saved


In [68]:
high_music = pd.read_csv("/content/high_popularity_spotify_data.csv")
low_music = pd.read_csv("/content/low_popularity_spotify_data.csv")

music = pd.concat(
    [high_music, low_music],
    ignore_index=True
)

music.head()

,energy,tempo,danceability,playlist_genre,loudness,liveness,valence,track_artist,time_signature,speechiness,...,instrumentalness,track_album_id,mode,key,duration_ms,acousticness,id,playlist_subgenre,type,playlist_id
0,0.592,157.969,0.521,pop,-7.777,0.122,0.535,"Lady Gaga, Bruno Mars",3.0,0.0304,...,0.0000,10FLjwfpbxLmW8c25Xyc2N,0.0,6.0,251668.0,0.3080,2plbrEY59IikOBgBGLjaoe,mainstream,audio_features,37i9dQZF1DXcBWIGoYBM5M
1,0.507,104.978,0.747,pop,-10.171,0.117,0.438,Billie Eilish,4.0,0.0358,...,0.0608,7aJuG4TFXa2hmE4z1yxc3n,1.0,2.0,210373.0,0.2000,6dOtVTDdiauQNBQEDOtlAB,mainstream,audio_features,37i9dQZF1DXcBWIGoYBM5M
2,0.808,108.548,0.554,pop,-4.169,0.159,0.372,Gracie Abrams,4.0,0.0368,...,0.0000,0hBRqPYPXhr1RkTDG3n4Mk,1.0,1.0,166300.0,0.2140,7ne4VBA60CxGM75vw0EYad,mainstream,audio_features,37i9dQZF1DXcBWIGoYBM5M
3,0.910,112.966,0.670,pop,-4.070,0.304,0.786,Sabrina Carpenter,4.0,0.0634,...,0.0000,4B4Elma4nNDUyl6D5PvQkj,0.0,0.0,157280.0,0.0939,1d7Ptw3qYcfpdLNL5REhtJ,mainstream,audio_features,37i9dQZF1DXcBWIGoYBM5M
4,0.783,149.027,0.777,pop,-4.477,0.355,0.939,"ROSÉ, Bruno Mars",4.0,0.2600,...,0.0000,2IYQwwgxgOIn7t3iF6ufFD,0.0,0.0,169917.0,0.0283,5vNRhkKd0yEAg8suGBpjeY,mainstream,audio_features,37i9dQZF1DXcBWIGoYBM5M


In [69]:
music["track_name"] = music["track_name"].fillna("")
music["track_artist"] = music["track_artist"].fillna("")
music["playlist_genre"] = music["playlist_genre"].fillna("")

music["tags"] = (
    music["track_name"]
    + " "
    + music["track_artist"]
    + " "
    + music["playlist_genre"]
)

music["tags"] = music["tags"].str.lower()

In [70]:
music_df = music[
    [
        "track_name",
        "track_artist",
        "tags"
    ]
]

music_df.head()

,track_name,track_artist,tags
0,Die With A Smile,"Lady Gaga, Bruno Mars","die with a smile lady gaga, bruno mars pop"
1,BIRDS OF A FEATHER,Billie Eilish,birds of a feather billie eilish pop
2,That’s So True,Gracie Abrams,that’s so true gracie abrams pop
3,Taste,Sabrina Carpenter,taste sabrina carpenter pop
4,APT.,"ROSÉ, Bruno Mars","apt. rosé, bruno mars pop"


In [71]:
cv_music = CountVectorizer(
    max_features=3000,
    stop_words="english"
)

music_vectors = cv_music.fit_transform(
    music_df["tags"]
)

print(music_vectors.shape)

(4831, 3000)


In [72]:
music_dict = {
    "track_name": music_df["track_name"].tolist(),
    "track_artist": music_df["track_artist"].tolist()
}

pickle.dump(
    music_dict,
    open("music_dict.pkl", "wb")
)

print("music_dict.pkl Saved")

music_dict.pkl Saved


In [73]:
from scipy.sparse import save_npz

save_npz(
    "music_vectors.npz",
    music_vectors
)

pickle.dump(
    cv_music.vocabulary_,
    open("music_vocab.pkl", "wb")
)

print("music_vectors.npz Saved")

music_vectors.npz Saved


In [74]:
import os

files = [
    "movies_dict.pkl",
    "movie_vectors.npz",
    "movie_vocab.pkl",
    "music_dict.pkl",
    "music_vectors.npz",
    "music_vocab.pkl",
]

for file in files:
    print(f"{file} : {os.path.getsize(file) / (1024 * 1024):.2f} MB")

movies_dict.pkl : 0.08 MB
movie_vectors.npz : 0.32 MB
movie_vocab.pkl : 0.15 MB
music_dict.pkl : 0.16 MB
music_vectors.npz : 0.04 MB
music_vocab.pkl : 0.08 MB
